In [ ]:
# ─── Mount Google Drive ───
from google.colab import drive
drive.mount('/content/drive')

import os

# ⚙️ All outputs (checkpoints, plots, model) will be saved here on your Drive
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/ViT_Medical_FineTuning'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f'✅ Google Drive mounted. Outputs will be saved to: {DRIVE_OUTPUT_DIR}')

In [ ]:
# Install required packages
!pip install -q transformers>=4.35.0 accelerate datasets timm medmnist
!pip install -q scikit-learn matplotlib seaborn torchmetrics
!pip install -q opencv-python-headless

print('✅ All packages installed successfully!')

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms as transforms
from torchvision import datasets

# ✅ FIX: ViTFeatureExtractor was removed in transformers>=4.35
#         Use ViTImageProcessor instead (drop-in replacement)
from transformers import ViTForImageClassification, ViTImageProcessor
from transformers import get_cosine_schedule_with_warmup

import medmnist
from medmnist import INFO, Evaluator

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIGURATION — Edit these as needed
# ─────────────────────────────────────────────
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/ViT_Medical_FineTuning'  # synced with Step 1

CONFIG = {
    # Dataset: choose from 'bloodmnist', 'pathmnist', 'dermamnist', 'octmnist', 'pneumoniamnist'
    'dataset_name': 'bloodmnist',

    # ViT model: 'google/vit-base-patch16-224' (ViT-B/16)
    # For ViT-Large: 'google/vit-large-patch16-224'
    'model_name': 'google/vit-base-patch16-224',

    'image_size': 224,
    'batch_size': 32,          # Reduce to 16 if OOM
    'num_epochs': 15,
    'learning_rate': 2e-4,
    'weight_decay': 1e-2,
    'warmup_ratio': 0.1,
    'label_smoothing': 0.1,
    'dropout': 0.1,

    # Fine-tuning strategy
    'freeze_layers': 6,        # Freeze first N transformer blocks (0 = train all)
    'use_fp16': True,          # Mixed precision training
    'grad_clip': 1.0,

    'seed': 42,
    # Local fast scratch dir (Colab SSD) — periodically synced to Drive
    'output_dir': '/content/vit_medical_checkpoints',
    # Drive output dir (persistent)
    'drive_dir': DRIVE_OUTPUT_DIR,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(CONFIG['drive_dir'],  exist_ok=True)

# ─────────────────────────────────────────────
# Reproducibility
# ─────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(CONFIG['seed'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ─── Helper: save a file to both local /content and Google Drive ───
import shutil
def save_to_drive(local_path):
    """Copy a file from local scratch dir to Google Drive."""
    fname = os.path.basename(local_path)
    drive_path = os.path.join(CONFIG['drive_dir'], fname)
    shutil.copy2(local_path, drive_path)
    print(f'   💾 Saved to Drive: {drive_path}')

def sync_dir_to_drive(local_dir):
    """Copy all files in local_dir to Drive (non-recursive)."""
    for fname in os.listdir(local_dir):
        src = os.path.join(local_dir, fname)
        if os.path.isfile(src):
            dst = os.path.join(CONFIG['drive_dir'], fname)
            shutil.copy2(src, dst)
    print(f'✅ Synced {local_dir} → {CONFIG["drive_dir"]}')

print('\n✅ Config ready. Drive output dir:', CONFIG['drive_dir'])

# ─────────────────────────────────────────────
# LOGGING SETUP — writes to console + .log file
# ─────────────────────────────────────────────
import logging
import datetime

run_timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
LOG_FILE_LOCAL = os.path.join(CONFIG['output_dir'], f'training_{run_timestamp}.log')
LOG_FILE_DRIVE = os.path.join(CONFIG['drive_dir'],  f'training_{run_timestamp}.log')

logger = logging.getLogger('vit_medical')
logger.setLevel(logging.DEBUG)
logger.handlers = []  # clear handlers if cell is re-run

fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')

# File handler — full DEBUG-level log written to disk
fh = logging.FileHandler(LOG_FILE_LOCAL, mode='w', encoding='utf-8')
fh.setLevel(logging.DEBUG)
fh.setFormatter(fmt)
logger.addHandler(fh)

# Stream handler — INFO+ shown in Colab cell output
sh = logging.StreamHandler()
sh.setLevel(logging.INFO)
sh.setFormatter(fmt)
logger.addHandler(sh)

# ── Log the full run config immediately ──
logger.info('=' * 62)
logger.info('  ViT Medical Fine-Tuning — Run Started')
logger.info(f'  Timestamp : {run_timestamp}')
logger.info(f'  Device    : {DEVICE}')
if torch.cuda.is_available():
    logger.info(f'  GPU       : {torch.cuda.get_device_name(0)}')
    logger.info(f'  VRAM      : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
logger.info('─' * 62)
logger.info('  CONFIG:')
for k, v in CONFIG.items():
    logger.info(f'    {k:<22}: {v}')
logger.info('=' * 62)
print(f'\n📝 Log file (local) : {LOG_FILE_LOCAL}')
print(f'   Log file (Drive) : {LOG_FILE_DRIVE}')

In [ ]:
import medmnist
from medmnist import INFO

# ─── Dataset info ───
DATASET_NAME = CONFIG['dataset_name']
info = INFO[DATASET_NAME]
DataClass = getattr(medmnist, info['python_class'])

NUM_CLASSES = len(info['label'])
CLASS_NAMES = list(info['label'].values())
CLASS_NAMES = []
for name in info['label'].values():
    if 'immature granulocytes' in name:
        CLASS_NAMES.append('granulocyte')  # or 'immature granulocyte'
    else:
        CLASS_NAMES.append(name)
TASK = info['task']

print(f'📦 Dataset : {DATASET_NAME}')
print(f'📋 Task    : {TASK}')
print(f'🏷️  Classes : {NUM_CLASSES} → {CLASS_NAMES}')
print(f'🔗 Source  : {info["url"]}')

# ─── Transforms ───
# Mean/std from ImageNet (ViT was pretrained on ImageNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ─── Load splits ───
train_dataset = DataClass(split='train', transform=train_transform, download=True, as_rgb=True)
val_dataset   = DataClass(split='val',   transform=val_transform,   download=True, as_rgb=True)
test_dataset  = DataClass(split='test',  transform=val_transform,   download=True, as_rgb=True)

print(f'\n📈 Train samples : {len(train_dataset)}')
print(f'📉 Val samples   : {len(val_dataset)}')
print(f'🧪 Test samples  : {len(test_dataset)}')

In [ ]:
# ─── Visualize sample images ───
fig, axes = plt.subplots(3, 8, figsize=(18, 7))
fig.suptitle(f'Sample Images — {DATASET_NAME.upper()}', fontsize=15, fontweight='bold')

for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    # Denormalize
    img = img * torch.tensor(IMAGENET_STD).view(3,1,1) + torch.tensor(IMAGENET_MEAN).view(3,1,1)
    img = img.permute(1, 2, 0).numpy().clip(0, 1)
    ax.imshow(img)
    lbl = label.item() if hasattr(label, 'item') else int(label)
    ax.set_title(CLASS_NAMES[lbl], fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{CONFIG["output_dir"]}/sample_images.png', dpi=120, bbox_inches='tight')
save_to_drive(f'{CONFIG["output_dir"]}/sample_images.png')
plt.show()
print('✅ Sample images saved.')

In [ ]:
# ─── Class distribution ───
labels_all = [int(train_dataset[i][1]) for i in range(len(train_dataset))]
label_counts = np.bincount(labels_all)

plt.figure(figsize=(10, 4))
bars = plt.bar(CLASS_NAMES, label_counts, color=plt.cm.tab10.colors[:NUM_CLASSES], edgecolor='black', linewidth=0.5)
plt.title('Training Set — Class Distribution', fontsize=13, fontweight='bold')
plt.ylabel('Number of Samples')
plt.xticks(rotation=20, ha='right')
for bar, count in zip(bars, label_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, str(count),
             ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(f'{CONFIG["output_dir"]}/class_distribution.png', dpi=120, bbox_inches='tight')
save_to_drive(f'{CONFIG["output_dir"]}/class_distribution.png')
plt.show()

# ─── Weighted sampler to handle class imbalance ───
class_weights = 1.0 / torch.tensor(label_counts, dtype=torch.float)
sample_weights = [class_weights[lbl] for lbl in labels_all]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'], shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG['batch_size'], shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'✅ DataLoaders ready. Batches per epoch: {len(train_loader)}')
logger.info('─' * 62)
logger.info(f'Dataset         : {DATASET_NAME}  |  Task: {TASK}')
logger.info(f'Num classes     : {NUM_CLASSES}  →  {CLASS_NAMES}')
logger.info(f'Train samples   : {len(train_dataset)}')
logger.info(f'Val samples     : {len(val_dataset)}')
logger.info(f'Test samples    : {len(test_dataset)}')
logger.info(f'Batches/epoch   : {len(train_loader)}')
logger.info(f'Class counts    : {label_counts.tolist()}')

In [ ]:
from transformers import ViTForImageClassification, ViTConfig

print(f'⬇️  Loading checkpoint: {CONFIG["model_name"]} ...')

model = ViTForImageClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,   # Replace the head
    hidden_dropout_prob=CONFIG['dropout'],
    attention_probs_dropout_prob=CONFIG['dropout'],
    id2label={i: name for i, name in enumerate(CLASS_NAMES)},
    label2id={name: i for i, name in enumerate(CLASS_NAMES)},
)

print('✅ Checkpoint loaded!')
print(f'   Hidden size : {model.config.hidden_size}')
print(f'   Num layers  : {model.config.num_hidden_layers}')
print(f'   Num heads   : {model.config.num_attention_heads}')
print(f'   Patch size  : {model.config.patch_size}')
print(f'   New head    : {NUM_CLASSES} output classes')

# ─── Freeze early transformer blocks ───
freeze_n = CONFIG['freeze_layers']
if freeze_n > 0:
    # Freeze patch embeddings
    for param in model.vit.embeddings.parameters():
        param.requires_grad = False
    # Freeze first N encoder layers
    for i, layer in enumerate(model.vit.encoder.layer):
        if i < freeze_n:
            for param in layer.parameters():
                param.requires_grad = False
    print(f'🔒 Froze embeddings + first {freeze_n} encoder blocks')

# ─── Parameter count ───
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params
print(f'\n📊 Total params    : {total_params:,}')
print(f'   Trainable      : {trainable_params:,} ({100*trainable_params/total_params:.1f}%)')
print(f'   Frozen         : {frozen_params:,}')

model = model.to(DEVICE)
logger.info('─' * 62)
logger.info(f'Model           : {CONFIG["model_name"]}')
logger.info(f'Hidden size     : {model.config.hidden_size}')
logger.info(f'Encoder layers  : {model.config.num_hidden_layers}')
logger.info(f'Attn heads      : {model.config.num_attention_heads}')
logger.info(f'Patch size      : {model.config.patch_size}')
logger.info(f'Frozen layers   : {CONFIG["freeze_layers"]}')
logger.info(f'Total params    : {total_params:,}')
logger.info(f'Trainable params: {trainable_params:,}  ({100*trainable_params/total_params:.1f}%)')

In [ ]:
from torch.cuda.amp import GradScaler, autocast

# ─── Layer-wise learning rate decay (LLRD) ───
# Lower LR for earlier (more general) layers, higher for later (task-specific) layers
def get_parameter_groups(model, base_lr, weight_decay, llrd_factor=0.9):
    no_decay = ['bias', 'LayerNorm.weight']
    groups = []
    # Classifier head — highest LR
    groups.append({
        'params': [p for n, p in model.classifier.named_parameters() if p.requires_grad],
        'lr': base_lr, 'weight_decay': weight_decay
    })
    # LayerNorm + pooler
    groups.append({
        'params': [p for n, p in model.vit.layernorm.named_parameters() if p.requires_grad],
        'lr': base_lr * llrd_factor, 'weight_decay': 0.0
    })
    # Encoder layers — LLRD
    num_layers = len(model.vit.encoder.layer)
    for i, layer in reversed(list(enumerate(model.vit.encoder.layer))):
        layer_lr = base_lr * (llrd_factor ** (num_layers - i))
        wd_params = [p for n, p in layer.named_parameters()
                     if p.requires_grad and not any(nd in n for nd in no_decay)]
        no_wd_params = [p for n, p in layer.named_parameters()
                        if p.requires_grad and any(nd in n for nd in no_decay)]
        if wd_params:
            groups.append({'params': wd_params, 'lr': layer_lr, 'weight_decay': weight_decay})
        if no_wd_params:
            groups.append({'params': no_wd_params, 'lr': layer_lr, 'weight_decay': 0.0})
    return groups

param_groups = get_parameter_groups(model, CONFIG['learning_rate'], CONFIG['weight_decay'])
optimizer = optim.AdamW(param_groups, betas=(0.9, 0.999), eps=1e-8)

# ─── Cosine schedule with warmup ───
total_steps  = len(train_loader) * CONFIG['num_epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# ─── Loss ───
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])

# ─── Mixed precision scaler ───
scaler = GradScaler(enabled=CONFIG['use_fp16'])

print(f'✅ Optimizer  : AdamW with LLRD')
print(f'   Scheduler  : Cosine warmup ({warmup_steps} warmup / {total_steps} total steps)')
print(f'   Loss       : CrossEntropy (label_smoothing={CONFIG["label_smoothing"]})')
print(f'   Mixed FP16 : {CONFIG["use_fp16"]}')

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, scaler, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.squeeze().long().to(device, non_blocking=True)

        optimizer.zero_grad()
        with autocast(enabled=CONFIG['use_fp16']):
            outputs = model(pixel_values=images)
            loss = criterion(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / len(loader), correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.squeeze().long().to(device, non_blocking=True)

        with autocast(enabled=CONFIG['use_fp16']):
            outputs = model(pixel_values=images)
            loss = criterion(outputs.logits, labels)

        probs = torch.softmax(outputs.logits, dim=1)
        preds = probs.argmax(dim=1)

        running_loss += loss.item()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    return (running_loss / len(loader), correct / total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))


# ─────────────────────────────────────────────
# MAIN TRAINING LOOP
# ─────────────────────────────────────────────
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_ckpt_path = os.path.join(CONFIG['output_dir'], 'best_vit_medical.pt')

logger.info('─' * 62)
logger.info('Training started')
logger.info(f'Total epochs    : {CONFIG["num_epochs"]}')
logger.info(f'Total steps     : {total_steps}  |  Warmup: {warmup_steps}')
logger.info('─' * 62)
logger.info(f'{"Epoch":>5} | {"TrainLoss":>9} | {"TrainAcc":>8} | {"ValLoss":>8} | {"ValAcc":>7} | {"LR":>10} | Note')
logger.info('─' * 62)

print('🚀 Starting Training...\n')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7} | {"LR":>10}')
print('─' * 65)

for epoch in range(1, CONFIG['num_epochs'] + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, scaler, DEVICE)
    val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion, DEVICE)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    current_lr = scheduler.get_last_lr()[0]
    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_acc': val_acc}, best_ckpt_path)
        save_to_drive(best_ckpt_path)

    marker = '  ⭐ BEST' if is_best else ''
    note = 'BEST' if is_best else ''
    logger.info(f'{epoch:>5} | {train_loss:>9.4f} | {train_acc*100:>7.2f}% | '
                f'{val_loss:>8.4f} | {val_acc*100:>6.2f}%  | {current_lr:.2e} | {note}')
    # Flush log to disk every epoch so Drive copy is never stale
    for h in logger.handlers:
        h.flush()
    print(f'{epoch:>6} | {train_loss:>10.4f} | {train_acc*100:>8.2f}% | '
          f'{val_loss:>8.4f} | {val_acc*100:>6.2f}%  | {current_lr:.2e}{marker}')

print(f'\n✅ Training complete! Best Val Acc: {best_val_acc*100:.2f}%')
print(f'   Best checkpoint: {best_ckpt_path}')
logger.info('─' * 62)
logger.info(f'Training complete. Best Val Acc: {best_val_acc*100:.2f}%')
logger.info(f'Best checkpoint : {best_ckpt_path}')
# Sync log to Drive so it's safe even if session crashes before Step 15
for h in logger.handlers: h.flush()
shutil.copy2(LOG_FILE_LOCAL, LOG_FILE_DRIVE)
print(f'📝 Log synced to Drive: {LOG_FILE_DRIVE}')

In [ ]:
epochs_range = range(1, CONFIG['num_epochs'] + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ViT Fine-Tuning — Training Curves', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
axes[0].plot(epochs_range, history['val_loss'],   'r-s', label='Val Loss',   markersize=4)
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc', markersize=4)
axes[1].plot(epochs_range, [a*100 for a in history['val_acc']],   'r-s', label='Val Acc',   markersize=4)
axes[1].axhline(best_val_acc*100, color='green', linestyle='--', alpha=0.7, label=f'Best={best_val_acc*100:.2f}%')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CONFIG["output_dir"]}/training_curves.png', dpi=120, bbox_inches='tight')
save_to_drive(f'{CONFIG["output_dir"]}/training_curves.png')
plt.show()
print('✅ Training curves saved.')

In [ ]:
# Load best checkpoint
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f'✅ Loaded best checkpoint from epoch {ckpt["epoch"]} (val_acc={ckpt["val_acc"]*100:.2f}%)')

# Full evaluation on test set
test_loss, test_acc, y_pred, y_true, y_prob = evaluate(model, test_loader, criterion, DEVICE)

# ─── Core metrics ───
acc   = accuracy_score(y_true, y_pred)
f1_macro  = f1_score(y_true, y_pred, average='macro')
f1_weighted = f1_score(y_true, y_pred, average='weighted')
prec  = precision_score(y_true, y_pred, average='weighted')
rec   = recall_score(y_true, y_pred, average='weighted')

# ROC-AUC (multi-class OvR)
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
if NUM_CLASSES == 2:
    roc_auc = roc_auc_score(y_true, y_prob[:, 1])
else:
    roc_auc = roc_auc_score(y_true_bin, y_prob, multi_class='ovr', average='macro')

print('\n' + '='*50)
print('        📊 TEST SET PERFORMANCE REPORT')
print('='*50)
print(f'  Accuracy          : {acc*100:.2f}%')
print(f'  F1 (Macro)        : {f1_macro:.4f}')
print(f'  F1 (Weighted)     : {f1_weighted:.4f}')
print(f'  Precision (W)     : {prec:.4f}')
print(f'  Recall (W)        : {rec:.4f}')
print(f'  ROC-AUC (macro)   : {roc_auc:.4f}')
print(f'  Test Loss         : {test_loss:.4f}')
print('='*50)

cls_report_str = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
print('\n📋 Per-class Classification Report:')
print(cls_report_str)

# ─── Log all test metrics ───
logger.info('─' * 62)
logger.info('TEST SET EVALUATION')
logger.info(f'  Loaded checkpoint epoch : {ckpt["epoch"]}')
logger.info(f'  Test Loss               : {test_loss:.4f}')
logger.info(f'  Accuracy                : {acc*100:.2f}%')
logger.info(f'  F1 Macro                : {f1_macro:.4f}')
logger.info(f'  F1 Weighted             : {f1_weighted:.4f}')
logger.info(f'  Precision (Weighted)    : {prec:.4f}')
logger.info(f'  Recall (Weighted)       : {rec:.4f}')
logger.info(f'  ROC-AUC (macro OvR)     : {roc_auc:.4f}')
logger.info('─' * 62)
logger.info('Per-class Classification Report:')
for line in cls_report_str.splitlines():
    logger.info('  ' + line)

# ─── Save classification report as a separate text file ───
report_local = os.path.join(CONFIG['output_dir'], 'classification_report.txt')
report_drive = os.path.join(CONFIG['drive_dir'],  'classification_report.txt')
with open(report_local, 'w') as rf:
    rf.write(f'ViT Medical Fine-Tuning — Classification Report\n')
    rf.write(f'Run timestamp : {run_timestamp}\n')
    rf.write(f'Model         : {CONFIG["model_name"]}\n')
    rf.write(f'Dataset       : {DATASET_NAME}\n')
    rf.write('=' * 62 + '\n')
    rf.write(f'Accuracy      : {acc*100:.2f}%\n')
    rf.write(f'F1 Macro      : {f1_macro:.4f}\n')
    rf.write(f'F1 Weighted   : {f1_weighted:.4f}\n')
    rf.write(f'Precision (W) : {prec:.4f}\n')
    rf.write(f'Recall (W)    : {rec:.4f}\n')
    rf.write(f'ROC-AUC       : {roc_auc:.4f}\n')
    rf.write(f'Test Loss     : {test_loss:.4f}\n')
    rf.write('=' * 62 + '\n\n')
    rf.write(cls_report_str)
shutil.copy2(report_local, report_drive)
print(f'\n📄 Classification report saved → {report_drive}')

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Confusion Matrix — {DATASET_NAME.upper()} Test Set', fontsize=14, fontweight='bold')

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title('Raw Counts'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=30); axes[0].tick_params(axis='y', rotation=0)

# Normalized
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='RdYlGn', ax=axes[1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, vmin=0, vmax=1)
axes[1].set_title('Normalized (Recall per Class)'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=30); axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(f'{CONFIG["output_dir"]}/confusion_matrix.png', dpi=120, bbox_inches='tight')
save_to_drive(f'{CONFIG["output_dir"]}/confusion_matrix.png')
plt.show()
print('✅ Confusion matrix saved.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.tab10.colors

for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc_i = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{cls_name} (AUC={roc_auc_i:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='Random Classifier')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(f'ROC Curves — {DATASET_NAME.upper()} (Macro AUC={roc_auc:.4f})', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CONFIG["output_dir"]}/roc_curves.png', dpi=120, bbox_inches='tight')
save_to_drive(f'{CONFIG["output_dir"]}/roc_curves.png')
plt.show()
print('✅ ROC curves saved.')

In [ ]:
per_cls_prec = precision_score(y_true, y_pred, average=None)
per_cls_rec  = recall_score(y_true, y_pred, average=None)
per_cls_f1   = f1_score(y_true, y_pred, average=None)

x = np.arange(NUM_CLASSES)
width = 0.28

fig, ax = plt.subplots(figsize=(max(10, NUM_CLASSES * 1.2), 5))
b1 = ax.bar(x - width, per_cls_prec, width, label='Precision', color='steelblue',   alpha=0.85)
b2 = ax.bar(x,          per_cls_rec,  width, label='Recall',    color='darkorange',  alpha=0.85)
b3 = ax.bar(x + width,  per_cls_f1,   width, label='F1-Score',  color='forestgreen', alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=25, ha='right')
ax.set_ylim([0, 1.1]); ax.set_ylabel('Score'); ax.set_xlabel('Class')
ax.set_title('Per-Class Metrics — Test Set', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)

for bar in [b1, b2, b3]:
    for rect in bar:
        h = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2, h + 0.01, f'{h:.2f}',
                ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(f'{CONFIG["output_dir"]}/per_class_metrics.png', dpi=120, bbox_inches='tight')
save_to_drive(f'{CONFIG["output_dir"]}/per_class_metrics.png')
plt.show()
print('✅ Per-class metrics chart saved.')

In [ ]:
from sklearn.manifold import TSNE

# Extract CLS token embeddings from test set
model.eval()
all_feats, all_lbls = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        # Get hidden states from last encoder layer
        out = model.vit(pixel_values=images)
        cls_tokens = out.last_hidden_state[:, 0, :].cpu().numpy()  # CLS token
        all_feats.append(cls_tokens)
        all_lbls.extend(labels.squeeze().numpy())

all_feats = np.vstack(all_feats)
all_lbls  = np.array(all_lbls)

print(f'Embedding shape: {all_feats.shape}  →  Running t-SNE...')

tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42, verbose=1)
tsne_feats = tsne.fit_transform(all_feats)

# Plot
fig, ax = plt.subplots(figsize=(12, 9))
for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, plt.cm.tab10.colors)):
    mask = all_lbls == i
    ax.scatter(tsne_feats[mask, 0], tsne_feats[mask, 1],
               c=[color], label=cls_name, alpha=0.65, s=12)

ax.set_title('t-SNE of ViT CLS Token Embeddings (Test Set)', fontsize=13, fontweight='bold')
ax.legend(loc='best', fontsize=9, markerscale=2)
ax.axis('off')

plt.tight_layout()
plt.savefig(f'{CONFIG["output_dir"]}/tsne_embeddings.png', dpi=120, bbox_inches='tight')
save_to_drive(f'{CONFIG["output_dir"]}/tsne_embeddings.png')
plt.show()
print('✅ t-SNE plot saved.')

In [ ]:
print('\n' + '='*60)
print('  🏥 ViT MEDICAL FINE-TUNING — FINAL SUMMARY REPORT')
print('='*60)
print(f'  Model            : {CONFIG["model_name"]}')
print(f'  Dataset          : {DATASET_NAME.upper()} ({NUM_CLASSES} classes)')
print(f'  Image Size       : {CONFIG["image_size"]}×{CONFIG["image_size"]}')
print(f'  Epochs Trained   : {CONFIG["num_epochs"]}')
print(f'  Frozen Layers    : {CONFIG["freeze_layers"]} / {model.config.num_hidden_layers}')
print(f'  Trainable Params : {trainable_params:,}')
print('─'*60)
print(f'  BEST Val Acc     : {best_val_acc*100:.2f}%')
print('─'*60)
print(f'  Test Accuracy    : {acc*100:.2f}%')
print(f'  Test F1 (Macro)  : {f1_macro:.4f}')
print(f'  Test F1 (W)      : {f1_weighted:.4f}')
print(f'  Test Precision   : {prec:.4f}')
print(f'  Test Recall      : {rec:.4f}')
print(f'  Test ROC-AUC     : {roc_auc:.4f}')
print('─'*60)
print(f'  Checkpoint path  : {best_ckpt_path}')
print('='*60)

logger.info('─' * 62)
logger.info('FINAL SUMMARY')
logger.info(f'  Model            : {CONFIG["model_name"]}')
logger.info(f'  Dataset          : {DATASET_NAME.upper()} ({NUM_CLASSES} classes)')
logger.info(f'  Epochs Trained   : {CONFIG["num_epochs"]}')
logger.info(f'  Best Val Acc     : {best_val_acc*100:.2f}%')
logger.info(f'  Test Accuracy    : {acc*100:.2f}%')
logger.info(f'  Test F1 Macro    : {f1_macro:.4f}')
logger.info(f'  Test F1 Weighted : {f1_weighted:.4f}')
logger.info(f'  Test Precision   : {prec:.4f}')
logger.info(f'  Test Recall      : {rec:.4f}')
logger.info(f'  Test ROC-AUC     : {roc_auc:.4f}')
logger.info('─' * 62)

# ─── Flush & sync log to Drive ───
for h in logger.handlers: h.flush()
shutil.copy2(LOG_FILE_LOCAL, LOG_FILE_DRIVE)
print(f'\n📝 Log synced to Drive: {LOG_FILE_DRIVE}')

# List all saved files
print('\n📁 Saved outputs:')
for f in sorted(os.listdir(CONFIG['output_dir'])):
    fpath = os.path.join(CONFIG['output_dir'], f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'   {f:40s}  {size_kb:7.1f} KB')

In [ ]:
# Save HuggingFace model (can be pushed to Hub or loaded with from_pretrained)
hf_save_path = os.path.join(CONFIG['output_dir'], 'hf_model')
model.save_pretrained(hf_save_path)
print(f'✅ HuggingFace model saved locally: {hf_save_path}')

# ─── Copy HF model folder to Google Drive ───
drive_hf_path = os.path.join(CONFIG['drive_dir'], 'hf_model')
if os.path.exists(drive_hf_path):
    shutil.rmtree(drive_hf_path)
shutil.copytree(hf_save_path, drive_hf_path)
print(f'💾 HuggingFace model copied to Drive: {drive_hf_path}')

# ─── Final sync of all remaining files to Drive ───
sync_dir_to_drive(CONFIG['output_dir'])

# ─── List everything on Drive ───
print('\n📁 Files saved to Google Drive:')
for f in sorted(os.listdir(CONFIG['drive_dir'])):
    fpath = os.path.join(CONFIG['drive_dir'], f)
    if os.path.isfile(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f'   {f:45s}  {size_kb:8.1f} KB')
    else:
        print(f'   {f:45s}  [folder]')

print(f'\n🎉 All done! Everything saved to Google Drive: {CONFIG["drive_dir"]}')

# ─── Final log entry + close ───
logger.info('─' * 62)
logger.info('All outputs saved to Google Drive.')
logger.info(f'Drive directory : {CONFIG["drive_dir"]}')
logger.info('Run complete. Closing log.')
logger.info('=' * 62)
for h in logger.handlers:
    h.flush()
    h.close()
logger.handlers = []
# Final Drive sync of the log file
shutil.copy2(LOG_FILE_LOCAL, LOG_FILE_DRIVE)
print(f'\n📝 Final log saved to Drive: {LOG_FILE_DRIVE}')